In [3]:
import pandas as pd

df = pd.read_parquet(
    "s3://analytics-transformed-parquet-12891/output/"
)

df.head()

,session_id,total_events,total_button_clicks,converted,session_duration_sec,avg_distance_to_checkout,unique_elements_clicked,button_click_ratio
0,ft0vhwqzj6t,7,0,0,2,1411.540324,0,0.000000
1,kpiqoymjh8j,3,0,0,1,1303.147703,0,0.000000
2,cq1d9a4lyk,11,6,1,2,133.242439,3,0.545455
3,htxny22p9p,3,0,0,0,850.872547,0,0.000000
4,s1c15hv00cq,7,0,0,2,474.724002,0,0.000000


In [6]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [28]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

X = df.drop(
    columns=["session_id", "converted"]
)

y = df["converted"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [29]:
from sklearn.metrics import accuracy_score

y_pred = xgb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9090909090909091


In [30]:
import joblib

joblib.dump(
    xgb_model,
    "checkout_predictor.pkl"
)

['checkout_predictor.pkl']

In [33]:
import tarfile
import os
import boto3
import sagemaker

# 1. Save the model locally using the specific name SageMaker expects
model_filename = "xgboost-model.ubj"
xgb_model.save_model(model_filename)

# 2. Package the model into a model.tar.gz file
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(model_filename)

# 3. Upload to S3
s3 = boto3.client("s3")

bucket = "analytics-transformed-parquet-12891"
key = "models/checkout/model.tar.gz"

s3.upload_file("model.tar.gz", bucket, key)

model_s3_path = f"s3://{bucket}/{key}"
print(model_s3_path)

s3://analytics-transformed-parquet-12891/models/checkout/model.tar.gz


In [34]:
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.serve import ModelBuilder
from sagemaker.core import image_uris

role = get_execution_role()

# 1. Initialize ModelBuilder
model_builder = ModelBuilder(
    model_path=model_s3_path,
    image_uri=image_uris.retrieve("xgboost", region="ap-southeast-1", version="1.7-1"),
    role_arn=role # Pass role here
)

# 2. Build (Optional - creates the model resource in AWS)
model_builder.build()

# 3. CALL DEPLOY ON THE BUILDER, NOT THE MODEL
predictor = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge"
)

╭──────────────────────────────────────────────── Wait Log Panel ─────────────────────────────────────────────────╮
│ [    ] Waiting for Endpoint... 0:03:30                                                                          │
│ ⠴ Current status: Creating                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/02/26 08:59:43] INFO     ✅ Deployment successful: Endpoint 'endpoint-0a74440a' using     ]8;id=7414410;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=7414411;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2687\2687]8;;\
                             None in SAGEMAKER_ENDPOINT mode (ARN:                                                 
                             arn:aws:sagemaker:ap-southeast-1:314146331900:endpoint/endpoint-                      
                             0a74440a)                                                                             